# Sistema Automatizado de Coleta de Comentários no Google Maps

<!-- ### Informações importantes -->
Este projeto apresenta o desenvolvimento de um sistema computacional capaz de realizar a **coleta automatizada de comentários e avaliações disponíveis no Google Maps**.  
A solução foi implementada utilizando **Selenium WebDriver** para navegação automatizada, extraindo informações como identificador do comentário, usuário, data de publicação, texto, avaliação em estrelas, presença de foto e status de *Local Guide*.  

Os dados coletados são organizados em uma base estruturada e exportados em formato compatível com planilhas, permitindo sua posterior utilização em análises quantitativas e qualitativas.  
O sistema busca **otimizar o processo de obtenção de informações** em grande escala, reduzindo a necessidade de coleta manual, garantindo maior eficiência, confiabilidade e padronização dos resultados.

## 📖 Utilização
Para utilizar o sistema, é necessário fornecer o **link do local desejado no Google Maps**.  
O programa acessará automaticamente a página indicada, realizará a extração dos comentários disponíveis e os salvará em uma planilha estruturada.  

Etapas básicas de uso:
1. Informar o link do local do Google Maps a ser analisado.  
1. Executar o notebook.  
1. Aguardar a coleta automatizada dos comentários.  
1. Consultar os dados exportados na planilha gerada.

### Insira no campo abaixo, entre as aspas duplas a URL do local do Google Maps

In [120]:
web_scrapping_target = "https://maps.app.goo.gl/2XNTgtFvbRhMp1m29"
# web_scrapping_target = "https://www.google.com/maps/place/Posto+Ipiranga/@-23.6185106,-46.6071845,17.25z/data=!4m16!1m9!3m8!1s0x94ce5bf54ea3c9c9:0xf530f12d6d967cf9!2sAssa%C3%AD+Atacadista!8m2!3d-23.6183251!4d-46.6092761!9m1!1b1!16s%2Fg%2F11mxzpwghk!3m5!1s0x94ce5bc0925106a7:0x4a46fd8e45009475!8m2!3d-23.6180355!4d-46.6052243!16s%2Fg%2F1yfj9_h05?entry=ttu&g_ep=EgoyMDI1MDkwMi4wIKXMDSoASAFQAw%3D%3D"


## Coleta automática

### Importações

In [137]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import TimeoutException

import urllib.parse
import re
import pandas as pd
import os

from datetime import datetime, timedelta
from time import sleep

### Definição de variáveis globais e Funções

In [ ]:
# Instanciar navegador
service = Service(ChromeDriverManager().install())
navegador = webdriver.Chrome(service=service)

# definir tempo máximo de aguardo para espera explícita (explicit wait)
wait = WebDriverWait(navegador, 10)

In [123]:
# Variáveis
colunas_df = ["id", "usuario", "ano_postagem", "comentario", "avaliacao", "foto", "local_guide"]

objects_dom = {
    'navbar_avaliacoes': '/html/body/div[1]/div[2]/div[9]/div[8]/div/div/div[1]/div[2]/div/div[1]/div/div/div[3]/div/div/button[2]',
    'container_avaliacoes': '.m6QErb.DxyBCb.kA9KIf.dS8AEf.XiKgde',
    'btn_ordenar': '.TrU0dc.kdfrQc.NUqjXc',
    'btn_ordenar_recentes': '/html/body/div[1]/div[2]/div[4]/div[1]/div/div[2]',
    'container_grupo_comentarios': '/html/body/div[1]/div[2]/div[9]/div[8]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[9]',
    'container_comentario': '.jftiEf.fontBodyMedium',
    'final_pagina': '.lXJj5c.Hk4XGb',
    'obj_load': '.lXJj5c.Hk4XGb .uj8Yqe'
}

In [143]:
# funções
def coletar_dados_comentário(elemento_pai: WebElement):
    id = elemento_pai.get_attribute('data-review-id')
    usuario = elemento_pai.find_element(By.CSS_SELECTOR, '.d4r55').text
    avaliacao = elemento_pai.find_element(By.CSS_SELECTOR, '.kvMYJc').get_attribute('aria-label')
    texto_tempo_postagem = elemento_pai.find_element(By.CSS_SELECTOR, '.rsqaWe').text

    ano_postagem = aproximar_ano(texto_tempo_postagem, datetime.now())

    # Expandir botão de comentário (se tiver)
    try:
        botao_expandir = WebDriverWait(elemento_pai, 2).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".w8nwRe.kyuRq"))
        )
    
        botao_expandir.click()
    
        # Aguarda o botão sumir (confirma que expandiu)
        WebDriverWait(elemento_pai, 5).until(
            EC.invisibility_of_element(botao_expandir)
        )
    
        print("Comentário expandido")
    
    except TimeoutException:
        # Botão não existe OU não ficou clicável
        pass
            
    # Pegar comentário
    try:        
        comentario = elemento_pai.find_element(By.CSS_SELECTOR, '.MyEned').text
    except NoSuchElementException:
        comentario = 'Sem comentários.'    

    # Se tem foto
    try: 
        elemento_pai.find_element(By.CSS_SELECTOR, '.KtCyie')
        foto = True
    except NoSuchElementException:
        foto = False
        

    # Local guide
    try:
        if "Local Guide" in elemento_pai.find_element(By.CSS_SELECTOR, '.RfnDt').text:
            local_guide = True
        else:
            local_guide = False
    except NoSuchElementException:
        local_guide = False
        
    return {
        "id": id,
        "usuario": usuario,
        "ano_postagem": ano_postagem,
        "avaliacao": avaliacao,
        "comentario": comentario,
        "local_guide": local_guide,
        "foto": foto
    }


def aproximar_ano(texto: str, referencia: datetime = None) -> int:
    if referencia is None:
        referencia = datetime.now()

    texto = texto.lower().strip()
    texto = texto.replace("editado ", "")  # remove prefixo se existir

    unidades = {
        "ano": 365,
        "anos": 365,
        "mês": 30,
        "meses": 30,
        "semana": 7,
        "semanas": 7,
        "dia": 1,
        "dias": 1
    }

    match = re.match(r"(um|\d+)\s+(\w+)\s+atrás", texto)
    if match:
        quantidade = match.group(1)
        unidade = match.group(2)

        quantidade = 1 if quantidade == "um" else int(quantidade)

        if unidade in unidades:
            dias = quantidade * unidades[unidade]
            data_aproximada = referencia - timedelta(days=dias)
            return data_aproximada.year

    return referencia.year  # fallback: assume que é este ano



def gerar_nome_arquivo(url):
    # Decodifica a URL
    url_decodificada = urllib.parse.unquote(url)

    # Extrai o nome do local
    nome_local_match = re.search(r'/place/([^/@]+)', url_decodificada)
    nome_local = nome_local_match.group(1).replace('+', '').replace(' ', '') if nome_local_match else 'LocalDesconhecido'

    # Extrai o identificador único (place ID)
    id_match = re.search(r'!1s(0x[a-fA-F0-9]+:0x[a-fA-F0-9]+)', url_decodificada)
    place_id = id_match.group(1).replace(':', '_') if id_match else 'IDDesconhecido'

    # Data atual no formato dd-mm-YYYY
    data_atual = datetime.now().strftime('%d-%m-%Y')

    # Monta o nome do arquivo
    nome_arquivo = f"Comentarios_{nome_local}_{place_id}_{data_atual}.xlsx"
    return nome_arquivo

### Execução principal

In [144]:
"""
Aqui inicialmente, o programa acessará a pagina do local escolhido, 
entrará na página de comentários e colocará em ordem dos mais recentes.
"""
# Acessar página
navegador.get(web_scrapping_target)
sleep(1)
# entrar no campo de comentários
wait.until(EC.element_to_be_clickable((By.XPATH, objects_dom['navbar_avaliacoes']))).click()

# Aguardar até carregar o container de avaliações
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, objects_dom['container_avaliacoes'])))

# Ordernar em mais recentes
wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, objects_dom['btn_ordenar']))).click()
sleep(1)
wait.until(EC.element_to_be_clickable((By.XPATH, objects_dom['btn_ordenar_recentes']))).click()

# Esperar carregar comentários
navegador.execute_script("arguments[0].scrollIntoView({ block: 'center'});", navegador.find_element(By.CSS_SELECTOR, '.lXJj5c.Hk4XGb'))

proximo_comentario = 0


nome_arquivo = gerar_nome_arquivo(navegador.current_url)
if os.path.exists(nome_arquivo):
    df = pd.read_excel(nome_arquivo)
else:
    df = pd.DataFrame(columns=colunas_df)

In [147]:
proximo_comentario = 1126
# nome_arquivo = gerar_nome_arquivo(navegador.current_url)
# if os.path.exists(nome_arquivo):
#     df = pd.read_excel(nome_arquivo)
# else:
#     df = pd.DataFrame(columns=colunas_df)
# df

In [149]:
# Coletar comentários e armazenar temporarimente.
#Para exportar, execute os blocos abaixos.

# A cada execução, o sistema coleta entorno de 400 comentários, e armazena temporarimente.
# Executando de novo, coleta mais 300 comentários.
# Quando desejar, basta exportar em uma planilha Excel.

stopScrapping = False
for c in range(1):
    stopScrapping = False
    # df_temp = pd.DataFrame(columns=colunas_df)
    buffer_temp = []
    
    print('Scrapping...')
    elementos_comentarios = navegador.find_elements(By.CSS_SELECTOR, objects_dom['container_comentario'])
    quantidade_comentarios = len(elementos_comentarios)
    # print(f'Quantidade de comentários: {quantidade_comentarios}')
    
    for i in range(proximo_comentario, quantidade_comentarios): # Rodar os ultimos 10
        print(f'Pegando o comentário {i}')
        navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", elementos_comentarios[i])
        dados = coletar_dados_comentário(elementos_comentarios[i])
        
        if dados['ano_postagem'] <= 2016:
            stopScrapping = True
            break

        if dados['id'] not in df['id'].values:
            print('Valor NÃO repetido. adicionar')
            # df_temp = pd.concat([df_temp, pd.DataFrame([dados])], ignore_index=True)
            buffer_temp.append(dados)
        else:
            print('Valor repetido.')
        proximo_comentario = i + 1
        sleep(0.1)

    # Rolar até o final página e carregar mais comentários
    try:
        print('Rolando para o final da página')
        navegador.execute_script("arguments[0].scrollIntoView({ block: 'center'});", navegador.find_element(By.CSS_SELECTOR, objects_dom['final_pagina']))
        sleep(0.5)
    
        proximo_loop = False
        print('Tentando pegar o elemento load')
        elemento_load = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, objects_dom['obj_load'])))

        # Identificar se está carregando
        while True: 
            resp = navegador.execute_script("""
                var elem = arguments[0];
                var rect = elem.getBoundingClientRect();
                var windowHeight = (window.innerHeight || document.documentElement.clientHeight);
                var windowWidth = (window.innerWidth || document.documentElement.clientWidth);
        
                return (
                    rect.bottom > 0 &&
                    rect.right > 0 &&
                    rect.top < windowHeight &&
                    rect.left < windowWidth
                );
            """, elemento_load)

            # Estava carrgando e parou de carregar
            if resp == False:
                print('Parou de carregar')
                break
            # else:
            #    print('Carregando...')
            sleep(0.2)
    except NoSuchElementException:
        # NÃO TEM MAIS COMENTÁRIOS - NÃO TEM ELEMENTO DE LOAD
        print('Loading nn encontrado - SEM MAIS COMENTÁRIOS')
        stopScrapping = True
    except StaleElementReferenceException as erro:
        # print(erro)
        stopScrapping = False
    else:
        # Se não deu erro, apareceu o elemento de carregar, e ja finalizou,
        # Verificar se tem mais dados
        print('Loading concluído')
        
        quantidade_comentarios = len(navegador.find_elements(By.CSS_SELECTOR, objects_dom['container_comentario']))
        print(f'Quantidade de comentários: {quantidade_comentarios}')
        print(f'Próximo comentário: indice {proximo_comentario}')
        if quantidade_comentarios > proximo_comentario:
            # Existem mais comentários
            print('Tem mais comentários, continuar')
            pass
        else:
            # NÃO TEM MAIS COMENTÁRIOS
            print('Não tem mais comentários')
            stopScrapping = True


    # Salvar dados temporários nos dados finais
    if buffer_temp:
        print('Salvando dados da nova pesquisa na tabela')
        # print(buffer_temp)
        df_temp = pd.DataFrame(buffer_temp)
        df = pd.concat([df, df_temp], ignore_index=True)

    if stopScrapping:
        print('Stop scrapping')
        break
print('Finalizou loop')

Scrapping...
Pegando o comentário 1127
Rolando para o final da página
Loading nn encontrado - SEM MAIS COMENTÁRIOS
Stop scrapping
Finalizou loop


In [151]:
# Visualizar comentários coletados
# buffer_temp
display(df)
# print(proximo_comentario)

,id,usuario,ano_postagem,comentario,avaliacao,foto,local_guide
0,Ci9DQUlRQUNvZENodHljRjlvT2xsd2FqTk1WekpZVmw5eG...,manuel soares,2026,Sem comentários.,5 estrelas,False,True
1,Ci9DQUlRQUNvZENodHljRjlvT2pkdVZrazFhWFZJWHkxNV...,Gabriel Ramalho,2026,Sem comentários.,5 estrelas,False,True
2,Ci9DQUlRQUNvZENodHljRjlvT2twSVMydzRSa05QYlRKNl...,fernando silva,2026,"Boa estação de Metro, fica bem situado, muito ...",4 estrelas,False,True
3,Ci9DQUlRQUNvZENodHljRjlvT2w5Q1N6Y3pTMlppZUVGQl...,goncalo ferreira,2025,Sem comentários.,5 estrelas,False,True
4,Ci9DQUlRQUNvZENodHljRjlvT2pWcVUwNUtWbVJ6Tm1Wel...,Aswathy Rajan,2025,Boa estação de metrô.,5 estrelas,False,False
...,...,...,...,...,...,...,...
1122,ChdDSUhNMG9nS0VJQ0FnSUNnNk9XRV93RRAB,Carlos Faigl,2017,As máquinas de bilhetes são um inferno. Longas...,1 estrela,False,True
1123,ChdDSUhNMG9nS0VJQ0FnSUNRMmVfSWhRRRAB,Marco Sousa (Sousafiano),2017,Sem comentários.,5 estrelas,False,True
1124,ChdDSUhNMG9nS0VJQ0FnSURBcDdhQTlRRRAB,Afonso Ramos,2017,Sem comentários.,4 estrelas,False,True
1125,ChdDSUhNMG9nS0VJQ0FnSURnMk5DN3pBRRAB,Henrique Martins,2017,É das melhores estações de metro da cidade do ...,5 estrelas,False,False


### EXPORTAR BASE DE DADOS

In [152]:
display(df)
print("Nome do arquivo:", nome_arquivo)
df.to_excel(nome_arquivo, index=False)

,id,usuario,ano_postagem,comentario,avaliacao,foto,local_guide
0,Ci9DQUlRQUNvZENodHljRjlvT2xsd2FqTk1WekpZVmw5eG...,manuel soares,2026,Sem comentários.,5 estrelas,False,True
1,Ci9DQUlRQUNvZENodHljRjlvT2pkdVZrazFhWFZJWHkxNV...,Gabriel Ramalho,2026,Sem comentários.,5 estrelas,False,True
2,Ci9DQUlRQUNvZENodHljRjlvT2twSVMydzRSa05QYlRKNl...,fernando silva,2026,"Boa estação de Metro, fica bem situado, muito ...",4 estrelas,False,True
3,Ci9DQUlRQUNvZENodHljRjlvT2w5Q1N6Y3pTMlppZUVGQl...,goncalo ferreira,2025,Sem comentários.,5 estrelas,False,True
4,Ci9DQUlRQUNvZENodHljRjlvT2pWcVUwNUtWbVJ6Tm1Wel...,Aswathy Rajan,2025,Boa estação de metrô.,5 estrelas,False,False
...,...,...,...,...,...,...,...
1122,ChdDSUhNMG9nS0VJQ0FnSUNnNk9XRV93RRAB,Carlos Faigl,2017,As máquinas de bilhetes são um inferno. Longas...,1 estrela,False,True
1123,ChdDSUhNMG9nS0VJQ0FnSUNRMmVfSWhRRRAB,Marco Sousa (Sousafiano),2017,Sem comentários.,5 estrelas,False,True
1124,ChdDSUhNMG9nS0VJQ0FnSURBcDdhQTlRRRAB,Afonso Ramos,2017,Sem comentários.,4 estrelas,False,True
1125,ChdDSUhNMG9nS0VJQ0FnSURnMk5DN3pBRRAB,Henrique Martins,2017,É das melhores estações de metro da cidade do ...,5 estrelas,False,False


Nome do arquivo: Comentarios_Trindade_0xd2464fb98f40b97_0x5565e22fe55a9619_02-02-2026.xlsx


### Testes

In [142]:
# Testar dom

# container avaliações: 
# final do container: .lXJj5c.Hk4XGb
# Container comentarios: /html/body/div[1]/div[2]/div[9]/div[8]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[9]
# Container comentario: .jftiEf.fontBodyMedium .jJc9Ad


# objects_dom['container_avaliacoes']
# wait.until(EC.element_to_be_clickable((By.XPATH, objects_dom['btn_ordenar_recentes'])))
# navegador.execute_script("arguments[0].scrollIntoView({ block: 'center'});", navegador.find_element(By.CSS_SELECTOR, objects_dom['final_pagina']))
# print(f'{objects_dom["container_comentario"]}:nth-child(4)')
# coletar_dados_comentário(navegador.find_element(By.XPATH, '/html/body/div[1]/div[2]/div[9]/div[8]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[9]/div[1049]'))